# Lab 11: Tangent polygons and Reuleaux variations

**Python supplement · Student notebook · English**

Two supplementary exercises, approximately 70–90 minutes in total. These can be assigned as homework or a separate computer session. Run setup and all helper cells in order. Complete the task functions, run the checks, and answer the discussion prompts. Sampling and floating-point checks illustrate the mathematical proofs in Sessions 10–11.

Geometry helpers are provided so that you can focus on support functions, constraints and variations.

Use [the notebook setup guide](README.md).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (9, 4), 'font.size': 11,
                     'axes.grid': True, 'grid.alpha': 0.25,
                     'figure.constrained_layout.use': True})

def closed(v):
    """Repeat the first vertex only for plotting."""
    return np.vstack([v, v[0]])

def area(v):
    """Unsigned shoelace area of an ordered simple polygon."""
    return abs(np.sum(v[:, 0]*np.roll(v[:, 1], -1)
                      - v[:, 1]*np.roll(v[:, 0], -1)))/2

def perimeter(v):
    return np.linalg.norm(np.roll(v, -1, axis=0)-v, axis=1).sum()

def outline(ax, v, **kwargs):
    p = closed(v)
    ax.plot(p[:, 0], p[:, 1], **kwargs)
    ax.set_aspect('equal', adjustable='box')

from scipy.spatial import ConvexHull, QhullError

from scipy.optimize import brentq


## Exercise 11.1: Refine tangent polygons and measure paired cuts

Allow 30–40 minutes. For equally spaced outward normals, the intersection
of neighboring support lines is
\[z_j=h_j u_j+\frac{h_{j+1}-h_j\cos\delta}{\sin\delta}u_j^\perp.\]
**Code tasks:** complete `tangent_polygon(h, N, alpha=pi/6)` using this
formula. It returns an ordered `(N,2)` vertex array; repeated vertices are
allowed at corners. Compare the disk, Reuleaux triangle and smooth body
`h=.5+.045*cos(3*theta)` for `N=6,12,24,48,96,192`.
These three examples share the same initial regular circumscribed hexagon.

Plot a refinement and the area errors. Verify `P=N*tan(pi/N)`, the ordering
of the three polygon areas, and the monotonicity of the gap between the
smooth body and the Reuleaux triangle. Check the **analytic** 6-to-12 area
losses using `s=2/sqrt(3)-1`: `3*sqrt(3)*s**2/2` for the disk and twice
that for the triangle. These are tangent polygons, so they generally do
not have constant width between the chosen normal directions.


**Provided helpers.** Run before completing the task.

In [ ]:
def directions(theta):
    theta=np.asarray(theta)
    return np.column_stack((np.cos(theta),np.sin(theta)))

def regular_centers(n):
    if n<3 or n%2!=1:
        raise ValueError('Use an odd number of vertices, at least three.')
    return directions(2*np.pi*np.arange(n)/n)/(2*np.cos(np.pi/(2*n)))

def arc_data(X):
    n=len(X); k=n//2; indices=np.arange(n)
    a=X[(indices+k)%n]-X
    b=X[(indices+k+1)%n]-X
    starts=np.arctan2(a[:,1],a[:,0])
    angles=np.mod(np.arctan2(b[:,1],b[:,0])-starts,2*np.pi)
    return starts,angles

def reuleaux_boundary(X, samples=120):
    starts,angles=arc_data(X)
    return np.vstack([c+directions(start+angle*np.arange(samples)/samples)
                      for c,start,angle in zip(X,starts,angles)])

def reuleaux_support(X,theta):
    # Exact maximization over each circular arc, plus its endpoints.
    theta=np.atleast_1d(theta); u=directions(theta)
    values=(X@u.T).max(axis=0)
    starts,angles=arc_data(X)
    for c,start,angle in zip(X,starts,angles):
        on_arc=np.mod(theta-start,2*np.pi)<=angle+1e-13
        values=np.maximum(values,np.where(on_arc,1+c@u.T,-np.inf))
    return values

def reuleaux_area(X):
    _,angles=arc_data(X)
    return area(X)+.5*np.sum(angles-np.sin(angles))

def check_reuleaux(X):
    n=len(X); k=n//2
    d=np.linalg.norm(X[:,None,:]-X[None,:,:],axis=2)
    np.testing.assert_allclose(d[np.arange(n),(np.arange(n)+k)%n],1.,atol=2e-12)
    assert d.max()<=1+2e-12, 'A diameter inequality failed.'
    _,angles=arc_data(X)
    assert np.all(angles>1e-7) and np.all(angles<np.pi/3+2e-12)
    np.testing.assert_allclose(angles.sum(),np.pi,atol=2e-12)
    q=np.linspace(0,2*np.pi,360,endpoint=False)
    np.testing.assert_allclose(reuleaux_support(X,q)+reuleaux_support(X,q+np.pi),1.,atol=2e-12)
    assert len(ConvexHull(X).vertices)==n

def polygon_sum(P,Q):
    pairs=(P[:,None,:]+Q[None,:,:]).reshape(-1,2)
    return pairs[ConvexHull(pairs).vertices]

def harmonic_data(theta, amplitudes, shift=(0.,0.)):
    theta=np.asarray(theta)
    h=np.full_like(theta,.5,dtype=float)
    hp=np.zeros_like(theta,dtype=float)
    rho=np.full_like(theta,.5,dtype=float)
    for frequency,coefficient in amplitudes.items():
        h+=coefficient*np.cos(frequency*theta)
        hp-=frequency*coefficient*np.sin(frequency*theta)
        rho+=(1-frequency**2)*coefficient*np.cos(frequency*theta)
    b,c=shift
    h+=b*np.cos(theta)+c*np.sin(theta)
    hp+=-b*np.sin(theta)+c*np.cos(theta)
    return h,hp,rho


In [ ]:
def tangent_polygon(h,N,alpha=np.pi/6):
    # TODO: intersect each pair of consecutive support lines.
    return None


**Run, visualize and check.**

In [ ]:
T=regular_centers(3)
hR=lambda q: reuleaux_support(T,q)
hS=lambda q: harmonic_data(q,{3:.045})[0]
hD=lambda q: np.full_like(np.asarray(q),.5,dtype=float)
result=tangent_polygon(hR,6)
if result is None:
    print('Complete tangent_polygon, then rerun this cell.')
else:
    counts=np.array([6,12,24,48,96,192])
    hs=[hR,hS,hD]; labels=['Reuleaux triangle','Smooth example','Disk']
    exact=np.array([(np.pi-np.sqrt(3))/2,np.pi/4-4*np.pi*.045**2,np.pi/4])
    measured=np.array([[area(tangent_polygon(h,N)) for h in hs] for N in counts])
    for j,h in enumerate(hs):
        for N in counts:
            P=tangent_polygon(h,N)
            np.testing.assert_allclose(perimeter(P),N*np.tan(np.pi/N),atol=2e-11)
        assert np.all(np.diff(measured[:,j])<=1e-12)
        assert np.all(measured[:,j]>=exact[j]-1e-12)
    assert np.all(np.diff(measured,axis=1)>=-1e-12)
    assert np.all(np.diff(measured[:,1]-measured[:,0])>=-1e-12)
    s=2/np.sqrt(3)-1
    np.testing.assert_allclose(measured[0,2]-measured[1,2],3*np.sqrt(3)*s*s/2,atol=1e-12)
    np.testing.assert_allclose(measured[0,0]-measured[1,0],3*np.sqrt(3)*s*s,atol=1e-12)
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for j,(h,label) in enumerate(zip(hs,labels)):
        outline(axes[0],tangent_polygon(h,12),label=label)
        axes[1].loglog(counts,measured[:,j]-exact[j],'-o',label=label)
    axes[0].set_title('Twelve prescribed tangent directions')
    axes[1].set(xlabel='number of normal directions',ylabel='outer polygon area minus exact area')
    for ax in axes: ax.legend(fontsize=8)
    plt.show()
    print('N; Reuleaux, smooth, disk areas')
    for N,row in zip(counts,measured): print(N,np.round(row,9))


**Explain your observations.** How can a nominal 12-sided tangent polygon have only nine nonzero sides? Why does the nondecreasing gap establish more than strict inequality at each mesh size?

*Write your response here.*

## Exercise 11.2: Check a Reuleaux-pentagon gradient and an admissible descent

Allow 40–50 minutes. Label the five centers counterclockwise, so `k=2`.
When `x0` changes to `x0+t*v`, recompute `xk` as the unit-circle intersection
about `x0+t*v` and `x[-1]`, on the branch near its old position; recompute
`x[k+1]` using `x0+t*v` and `x[1]`. The other centers remain fixed.
Helpers implement this geometry and the exact arc-plus-skeleton area.

**Code tasks:** complete `vertex_gradient(X)` from
\[g_0=(\tan(\theta_k/2)-\tan(\theta_0/2))(x_0-x_k)
 +(\tan(\theta_{k+1}/2)-\tan(\theta_0/2))(x_0-x_{k+1}).\]
It returns a two-component vector. Then complete `descent_trial(X)`:
try steps `0.02 / 2**j`, for `j=0,...,19`, along `-g0/||g0||`, accept
only a feasible polygon with strictly smaller area, and return
`(new_centers, accepted_step)`. Return `(X.copy(), 0.)` if no step is accepted
or the gradient vanishes. Never accept a failed feasibility check.

Compare the gradient with centered differences at a slightly perturbed
pentagon. Plot the accepted motion. Finally scan small displacements along
the inward radial direction of the **regular** pentagon. Check that its
first derivative vanishes while its second derivative is negative. This
single local descent is not a global optimizer; moving through vertex
collisions would require changing the polygon representation.

Reference: [Bogosel, variational arguments, Sections 3–4](https://arxiv.org/abs/2412.13808).


**Provided helpers.** Run before completing the task.

In [ ]:
def circle_intersection_near(a,b,reference):
    delta=b-a; d=np.linalg.norm(delta)
    if not 1e-10<d<2-1e-10:
        raise ValueError('Circle branch is degenerate.')
    middle=(a+b)/2
    perpendicular=np.array([-delta[1],delta[0]])/d
    offset=np.sqrt(1-d*d/4)*perpendicular
    choices=np.array([middle+offset,middle-offset])
    return choices[np.argmin(np.linalg.norm(choices-reference,axis=1))]

def perturb_vertex(X,v,t):
    n=len(X); k=n//2
    if n<5: raise ValueError('This local motion requires at least five vertices.')
    Y=X.copy(); Y[0]=X[0]+t*np.asarray(v)
    Y[k]=circle_intersection_near(Y[0],X[-1],X[k])
    Y[k+1]=circle_intersection_near(Y[0],X[1],X[k+1])
    return Y


In [ ]:
def vertex_gradient(X):
    # TODO: use arc_data(X) to get angles indexed by their centers.
    return None

def descent_trial(X):
    # TODO: backtrack along the normalized negative gradient.
    return None


**Run, visualize and check.**

In [ ]:
X0=regular_centers(5)
X=perturb_vertex(X0,np.array([-.8,.6]),.015)
check_reuleaux(X)
result=vertex_gradient(X)
trial=descent_trial(X)
if result is None or trial is None:
    print('Complete vertex_gradient and descent_trial, then rerun this cell.')
else:
    for n in (3,5,7,9):
        C=regular_centers(n); check_reuleaux(C)
        np.testing.assert_allclose(reuleaux_area(C),np.pi/2-n/2*np.tan(np.pi/(2*n)),atol=1e-12)
    eps=1e-5; basis=np.eye(2)
    finite=np.array([(reuleaux_area(perturb_vertex(X,v,eps))
                      -reuleaux_area(perturb_vertex(X,v,-eps)))/(2*eps) for v in basis])
    np.testing.assert_allclose(result,finite,atol=2e-8,rtol=2e-6)
    Y,step=trial; check_reuleaux(Y)
    assert step>0 and reuleaux_area(Y)<reuleaux_area(X)
    np.testing.assert_allclose(vertex_gradient(X0),0.,atol=1e-12)
    inward=-X0[0]/np.linalg.norm(X0[0])
    ts=np.linspace(-.025,.025,51)
    values=[]
    for t in ts:
        Z=perturb_vertex(X0,inward,t); check_reuleaux(Z)
        values.append(reuleaux_area(Z))
    values=np.array(values)
    eps2=2e-4
    second=(reuleaux_area(perturb_vertex(X0,inward,eps2))
            -2*reuleaux_area(X0)+reuleaux_area(perturb_vertex(X0,inward,-eps2)))/eps2**2
    angle=np.pi/5
    exact_second=2*(np.sin(angle/2)*np.sin(angle)-np.cos(3*angle/2))/(np.cos(angle/2)*np.sin(angle))
    np.testing.assert_allclose(second,exact_second,atol=2e-5)
    assert second<0
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    outline(axes[0],reuleaux_boundary(X),label='before')
    outline(axes[0],reuleaux_boundary(Y),label='accepted descent')
    axes[0].scatter(X[:,0],X[:,1],s=15)
    for i in (0,2,3):
        axes[0].annotate('',xy=Y[i],xytext=X[i],arrowprops=dict(arrowstyle='->',color='black'))
    axes[0].set_title('Three centers move together'); axes[0].legend(fontsize=8)
    axes[1].plot(ts,values-reuleaux_area(X0),'-o',ms=2)
    axes[1].axhline(0,color='black',lw=.8)
    axes[1].set(xlabel='inward displacement of vertex 0',ylabel='area minus regular pentagon area',
                title='Stationary, but not a local minimum')
    plt.show()
    print('Analytic gradient:',result,'; centered differences:',finite)
    print(f'Accepted step {step:.5f}: area {reuleaux_area(X):.9f} -> {reuleaux_area(Y):.9f}')
    print(f'Second derivative: finite differences {second:.6f}, analytic {exact_second:.6f}')


**Explain your observations.** Why would finite differences with just x0 moved give the wrong derivative? Why is the regular pentagon stationary although the area can decrease? Which geometric conditions would fail at a large step?

*Write your response here.*

**Before submitting:** restart the kernel and run all cells. Label the plots and distinguish analytic identities, sampling errors, feasibility checks and global conclusions.